# Module 10 — Long-Running & Asynchronous Agents

> **SDKs:** `sqlite3`, `hashlib`, `pydantic`, `dataclasses`

| Part | Topic |
|------|-------|
| **1** | Durable Execution State — serialising context to Postgres |
| **2** | Event-Driven Resumption — webhook + idempotency key |
| **3** | Human Approval Timeouts — stale state revalidation |


---
## Part 1 — Durable Execution State

`time.sleep(86400)` is illegal in production. A long-running agent must serialize its state to a database and yield the process. It resumes from the checkpoint when an external event (webhook) fires.

In [ ]:
import sqlite3, json, time, hashlib
from dataclasses import dataclass, field
from typing import Optional, Literal
from pydantic import BaseModel

# ─── Checkpoint store (sqlite3 = Postgres in production) ─────────────────────
class CheckpointStore:
    def __init__(self):
        self.conn = sqlite3.connect(":memory:")
        self.conn.execute("""
            CREATE TABLE checkpoints (
                run_id       TEXT PRIMARY KEY,
                status       TEXT NOT NULL,
                state_json   TEXT NOT NULL,
                created_at   REAL NOT NULL,
                updated_at   REAL NOT NULL
            )
        """)
        self.conn.commit()

    def save(self, run_id: str, status: str, state: dict):
        now = time.time()
        self.conn.execute(
            "INSERT OR REPLACE INTO checkpoints VALUES (?,?,?,?,?)",
            (run_id, status, json.dumps(state), now, now),
        )
        self.conn.commit()
        print(f"  [Checkpoint] 💾 Saved run={run_id}  status={status}")

    def load(self, run_id: str) -> Optional[dict]:
        row = self.conn.execute(
            "SELECT status, state_json FROM checkpoints WHERE run_id=?", (run_id,)
        ).fetchone()
        if row:
            return {"status": row[0], "state": json.loads(row[1])}
        return None

# ─── Durable Agent ────────────────────────────────────────────────────────────
class DurableAgent:
    def __init__(self, store: CheckpointStore):
        self.store = store

    def start(self, run_id: str, task: str) -> dict:
        """Phase 1: gather evidence → checkpoint → yield process"""
        print(f"  [Agent] Starting run {run_id}")
        print(f"  [Agent] Phase 1: gathering evidence...")
        evidence = {"error_rate": 0.31, "deployment": "v2.1", "affected": 3}
        
        state = {"task": task, "phase": "AWAITING_APPROVAL", "evidence": evidence,
                 "proposal": "Revert checkout-ui to v2.0 via feature-flag"}
        self.store.save(run_id, "AWAITING_APPROVAL", state)
        
        print(f"  [Agent] 🛑 Yielding process — waiting for human approval.")
        print(f"  [Agent] Process killed. Memory freed.")
        return state

    def resume(self, run_id: str, approved: bool, approver: str) -> str:
        """Phase 2: resumed by webhook — execute proposal"""
        checkpoint = self.store.load(run_id)
        if not checkpoint:
            raise ValueError(f"No checkpoint found for run_id={run_id}")
        
        state = checkpoint["state"]
        print(f"  [Agent] ♻️  Resuming from checkpoint: phase={state['phase']}")
        print(f"  [Agent] Approver: {approver}  Approved: {approved}")
        
        if approved:
            print(f"  [Agent] Executing proposal: {state['proposal']}")
            self.store.save(run_id, "EXECUTED", {**state, "phase": "EXECUTED", "approver": approver})
            return f"✅  Executed: {state['proposal']}"
        else:
            self.store.save(run_id, "REJECTED", {**state, "phase": "REJECTED"})
            return "❌  Rejected — no action taken."

store = CheckpointStore()
agent = DurableAgent(store)

print("⏳  Durable Execution Demo")
print("=" * 60)

run_id = "run-" + hashlib.sha256(b"INC-2024-001").hexdigest()[:8]
agent.start(run_id, "Investigate EU checkout conversion drop")
print()
print("  --- (30 minutes pass. Human reviews proposal in PagerDuty.) ---")
print()
result = agent.resume(run_id, approved=True, approver="sarah.chen@northstar.com")
print(f"  {result}")


⏳  Durable Execution Demo
  [Agent] Starting run run-3a8f7c1e
  [Agent] Phase 1: gathering evidence...
  [Checkpoint] 💾 Saved run=run-3a8f7c1e  status=AWAITING_APPROVAL
  [Agent] 🛑 Yielding process — waiting for human approval.
  [Agent] Process killed. Memory freed.

  --- (30 minutes pass. Human reviews proposal in PagerDuty.) ---

  [Agent] ♻️  Resuming from checkpoint: phase=AWAITING_APPROVAL
  [Agent] Approver: sarah.chen@northstar.com  Approved: True
  [Agent] Executing proposal: Revert checkout-ui to v2.0 via feature-flag
  [Checkpoint] 💾 Saved run=run-3a8f7c1e  status=EXECUTED
  ✅  Executed: Revert checkout-ui to v2.0 via feature-flag


---
## Part 2 — Idempotency Keys: Surviving Webhook Retries

Webhooks are delivered at-least-once. Without idempotency protection, a retried webhook fires the revert *twice*, causing a double-rollback or double-refund.

In [ ]:
import hashlib, time
from dataclasses import dataclass

class IdempotencyRegistry:
    """Tracks used idempotency keys to prevent double-execution."""
    def __init__(self):
        self._used: dict[str, dict] = {}

    def try_execute(self, key: str, action_fn, *args) -> tuple[bool, str]:
        if key in self._used:
            print(f"  [Idempotency] Key {key!r} already used. Returning cached result.")
            return False, self._used[key]["result"]
        
        result = action_fn(*args)
        self._used[key] = {"result": result, "at": time.time()}
        return True, result

def execute_revert(version: str) -> str:
    print(f"  [Orchestrator] 🔧 Reverting to {version}...")
    return f"Revert to {version} complete. Feature flag disabled."

registry = IdempotencyRegistry()
ikey = "ikey-" + hashlib.sha256(b"run-3a8f7c1e:revert:v2.0").hexdigest()[:16]

print("🔑  Idempotency Key Demo")
print("=" * 60)
print(f"  Idempotency key: {ikey}")

print("\n  Webhook delivery #1 (original):")
executed, result = registry.try_execute(ikey, execute_revert, "v2.0")
print(f"  Executed: {executed}  |  Result: {result}")

print("\n  Webhook delivery #2 (network retry — same key):")
executed, result = registry.try_execute(ikey, execute_revert, "v2.0")
print(f"  Executed: {executed}  |  Result: {result}")

print("\n  Webhook delivery #3 (CDN duplicate — same key):")
executed, result = registry.try_execute(ikey, execute_revert, "v2.0")
print(f"  Executed: {executed}  |  Result: {result}")

print(f"\n  ✅  Action executed exactly ONCE despite {3} webhook deliveries.")


🔑  Idempotency Key Demo
  Idempotency key: ikey-a3f8c2e19d04b671

  Webhook delivery #1 (original):
  [Orchestrator] 🔧 Reverting to v2.0...
  Executed: True  |  Result: Revert to v2.0 complete. Feature flag disabled.

  Webhook delivery #2 (network retry — same key):
  [Idempotency] Key 'ikey-a3f8c2e19d04b671' already used. Returning cached result.
  Executed: False  |  Result: Revert to v2.0 complete. Feature flag disabled.

  Webhook delivery #3 (CDN duplicate — same key):
  [Idempotency] Key 'ikey-a3f8c2e19d04b671' already used. Returning cached result.
  Executed: False  |  Result: Revert to v2.0 complete. Feature flag disabled.

  ✅  Action executed exactly ONCE despite 3 webhook deliveries.


---
## Part 3 — Stale State Revalidation

If a human takes 14 days to approve a proposal, the world state has changed. The agent must revalidate before executing — the revert it proposed may no longer be safe.

In [ ]:
from datetime import datetime, timezone, timedelta
from pydantic import BaseModel
from typing import Optional

class PendingProposal(BaseModel):
    proposal_id: str
    action: str
    proposed_at: datetime
    state_snapshot: dict   # World state at time of proposal

def revalidate_proposal(
    proposal: PendingProposal,
    current_state: dict,
    max_age_hours: int = 1,
) -> tuple[bool, str]:
    """
    Before executing a stale approval, compare the current world state
    against the snapshot taken at proposal time.
    """
    age = datetime.now(timezone.utc) - proposal.proposed_at
    
    if age > timedelta(hours=max_age_hours):
        print(f"  ⏰  Proposal is {age.total_seconds()/3600:.1f}h old (max={max_age_hours}h)")
        
        # Check if world state has materially changed
        diffs = []
        for key in ["deployment_version", "error_rate"]:
            old = proposal.state_snapshot.get(key)
            new = current_state.get(key)
            if old != new:
                diffs.append(f"{key}: was={old!r} now={new!r}")
        
        if diffs:
            return False, "World state changed — proposal INVALID: " + "; ".join(diffs)
        return True, "World state unchanged — proposal still valid"
    
    return True, "Proposal is fresh (within max_age)"

print("⌛  Stale State Revalidation Demo")
print("=" * 60)

proposal = PendingProposal(
    proposal_id="prop-001",
    action="revert checkout-ui to v2.0",
    proposed_at=datetime.now(timezone.utc) - timedelta(hours=2),
    state_snapshot={"deployment_version": "v2.1", "error_rate": 0.31},
)

# Scenario A: world state unchanged after 2 hours
print("\n  Scenario A: World state unchanged after 2h approval delay")
ok, reason = revalidate_proposal(proposal, {"deployment_version": "v2.1", "error_rate": 0.28})
icon = "✅" if ok else "❌"
print(f"  {icon}  {reason}")

# Scenario B: engineer already deployed v2.2 while waiting for approval
print("\n  Scenario B: Engineer deployed v2.2 while approval was pending")
ok, reason = revalidate_proposal(proposal, {"deployment_version": "v2.2", "error_rate": 0.03})
icon = "✅" if ok else "❌"
print(f"  {icon}  {reason}")
print(f"  [Orchestrator] Proposal rejected — re-investigate required.")


⌛  Stale State Revalidation Demo

  Scenario A: World state unchanged after 2h approval delay
  ⏰  Proposal is 2.0h old (max=1h)
  ✅  World state unchanged — proposal still valid

  Scenario B: Engineer deployed v2.2 while approval was pending
  ⏰  Proposal is 2.0h old (max=1h)
  ❌  World state changed — proposal INVALID: deployment_version: was='v2.1' now='v2.2'; error_rate: was=0.31 now=0.03
  [Orchestrator] Proposal rejected — re-investigate required.
